# Building a computer vision model for 100 food images
## TODO

* Model 1 - EfficientNetB4 from tf.
keras.applications
* Model 2 - https://www.tensorflow.org/tutorials/images/classification

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import tensorflow as tf

In [ ]:
!gcloud auth login

In [ ]:
from pathlib import Path
import shutil
import subprocess
import zipfile

ZIP_PATH = Path("100_whole_foods.zip")
DATASET_DIR = Path("100_whole_foods")
GCS_URI = "gs://food-vision-images-playground/100_whole_foods.zip"

subprocess.run(
    ["gcloud", "storage", "cp", GCS_URI, str(ZIP_PATH)],
    check=True,
)
print(f"Downloaded {ZIP_PATH.stat().st_size / 1e6:.1f} MB")

if not zipfile.is_zipfile(ZIP_PATH):
    raise RuntimeError(f"Downloaded file is not a complete zip: {ZIP_PATH}")

with zipfile.ZipFile(ZIP_PATH) as archive:
    bad_member = archive.testzip()
    if bad_member is not None:
        raise RuntimeError(f"Damaged file inside zip: {bad_member}")
    image_members = [
        name for name in archive.namelist()
        if Path(name).suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".gif"}
    ]
    if DATASET_DIR.exists():
        shutil.rmtree(DATASET_DIR)
    archive.extractall(Path.cwd())

if not (DATASET_DIR / "train").is_dir() or not (DATASET_DIR / "test").is_dir():
    raise RuntimeError("Zip must contain 100_whole_foods/train and 100_whole_foods/test")
print(f"Extracted {len(image_members)} images into {DATASET_DIR}")

In [ ]:
# !rm 100_whole_foods/.DS_Stor
!ls -la 100_whole_foods

In [ ]:
IMG_SIZE = (380, 380)
BATCH_SIZE = 32
SEED = 601
train_dir = "100_whole_foods/train"
test_dir = "100_whole_foods/test"

if not Path(train_dir).is_dir() or not Path(test_dir).is_dir():
    raise FileNotFoundError("Run the download and extraction cell first.")

print("Paths are ready. Datasets will be built after bad-image cleanup.")

# Image Checking

In [ ]:
from pathlib import Path
import tensorflow as tf

def find_bad_images(folder):
    bad_images = []

    valid_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".gif"}

    for path in Path(folder).rglob("*"):

        if path.is_file() and path.suffix.lower() in valid_extensions:

            try:
                # Check if file is empty
                if path.stat().st_size == 0:
                    bad_images.append((str(path), "EMPTY FILE"))
                    continue

                # Try reading the image
                image_bytes = tf.io.read_file(str(path))

                # Try decoding the image
                tf.io.decode_image(
                    image_bytes,
                    channels=3,
                    expand_animations=False
                )

            except Exception as e:
                bad_images.append((str(path), str(e)))

    return bad_images

In [ ]:
bad_train = find_bad_images("100_whole_foods/train")

print("Bad training images:", len(bad_train))

for path, error in bad_train:
    print(path)
    print(error)
    print()

In [ ]:
bad_test = find_bad_images("100_whole_foods/test")

print("Bad test images:", len(bad_test))

for path, error in bad_test:
    print(path)
    print(error)
    print()

In [ ]:
from pathlib import Path

# Delete bad training images
for path, error in bad_train:
    print("Deleting:", path)
    Path(path).unlink(missing_ok=True)

# Delete bad test images
for path, error in bad_test:
    print("Deleting:", path)
    Path(path).unlink(missing_ok=True)

In [ ]:
bad_train = find_bad_images(train_dir)
bad_test = find_bad_images(test_dir)
if bad_train or bad_test:
    raise RuntimeError("Bad images remain after cleanup. Run the cleanup cells again.")

# Build fresh datasets only after deleted files are gone.
train_data = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
)
test_data = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    label_mode="categorical",
    batch_size=BATCH_SIZE,
    shuffle=False,
)

class_names = train_data.class_names
if len(class_names) != 100:
    raise RuntimeError(f"Expected 100 classes, found {len(class_names)}")
if class_names != test_data.class_names:
    raise RuntimeError("Train and test class names do not match")

print("Clean datasets ready with 100 classes.")
class_names

# Prefetching Training and Testing Data

In [ ]:
# Speed up data loading where possible
train_data = train_data.prefetch(tf.data.AUTOTUNE)
test_data = test_data.prefetch(tf.data.AUTOTUNE)

train_data, test_data

In [ ]:
# 1. Create base model with tf.keras.applications
base_model = tf.keras.applications.efficientnet_v2.EfficientNetV2B0(include_top=False)

# OLD
# base_model = tf.keras.applications.EfficientNetB0(include_top=False)

# 2. Freeze the base model (so the pre-learned patterns remain)
base_model.trainable = False

# 3. Create inputs into the base model
inputs = tf.keras.layers.Input(shape=(380, 380, 3), name="input_layer")

# 5. Pass the inputs to the base_model (note: using tf.keras.applications, EfficientNetV2 inputs don't have to be normalized)
x = base_model(inputs)
# Check data shape after passing it to base_model
print(f"Shape after base_model: {x.shape}")

# 6. Average pool the outputs of the base model (aggregate all the most important information, reduce number of computations)
x = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pooling_layer")(x)
print(f"After GlobalAveragePooling2D(): {x.shape}")

# 7. Create the output activation layer
outputs = tf.keras.layers.Dense(len(class_names),
                                activation="softmax",
                                name="output_layer")(x)

# 8. Combine the inputs with the outputs into a model
model_0 = tf.keras.Model(inputs, outputs)

# 9. Compile the model
model_0.compile(loss='categorical_crossentropy',
                optimizer=tf.keras.optimizers.Adam(),
                metrics=["accuracy"])

# 10. Save the best complete model to Drive while training.
CHECKPOINT_DIR = Path("/content/drive/MyDrive/Nutrify/models")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
BEST_MODEL_PATH = CHECKPOINT_DIR / "food-100-version.keras"
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        BEST_MODEL_PATH,
        monitor="val_loss",
        save_best_only=True,
    ),
]

history = model_0.fit(
    train_data,
    epochs=25,
    validation_data=test_data,
    callbacks=callbacks,
)

In [ ]:
model_0.evaluate(test_data)

# Try the model on sample Image

In [ ]:
!wget https://raw.githubusercontent.com/PaingLinHtike/Nutrify/refs/heads/main/sample_food_images/chicken-wings.jpg

In [ ]:
import requests

url = 'https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcRKibimBB242syS7qS_eF4xmASSmccb4qorWWAbPH_T0TY5ffZfXRUIDjGM&s=10'
response = requests.get(url)

if response.status_code == 200:
    with open('downloaded_image.jpg', 'wb') as f:
        f.write(response.content)
    print('Image successfully downloaded as downloaded_image.jpg')
else:
    print(f'Failed to download image. Status code: {response.status_code}')

In [ ]:
from PIL import Image
img = Image.open("/content/downloaded_image.jpg")
img

In [ ]:
import numpy as np
import tensorflow as tf
image = tf.keras.utils.load_img("/content/downloaded_image.jpg")
input_arr = tf.keras.utils.img_to_array(image)
input_arr = np.array([input_arr]) # Convert single image to a batch.
input_arr = tf.image.resize(input_arr, size=(380, 380))
predictions = model_0.predict(input_arr)

In [ ]:
class_names[int(predictions.argmax(axis=1))]

# Save Nutrify Model 100_food

In [ ]:
import zipfile

# Save once more after training, then verify the file before downloading it.
MODEL_PATH = Path("/content/drive/MyDrive/Nutrify/models/food-100-version.keras")
model_0.save(MODEL_PATH)
if not zipfile.is_zipfile(MODEL_PATH):
    raise RuntimeError(f"Saved model is incomplete: {MODEL_PATH}")
print(f"Saved valid model: {MODEL_PATH} ({MODEL_PATH.stat().st_size / 1e6:.1f} MB)")

In [ ]:
import tensorflow as tf
import numpy as np

# Load the saved model and check its 100 outputs.
model_path = Path("/content/drive/MyDrive/Nutrify/models/food-100-version.keras")
loaded_model = tf.keras.models.load_model(model_path, compile=False)
if loaded_model.output_shape[-1] != 100:
    raise RuntimeError(f"Expected 100 outputs, found {loaded_model.output_shape[-1]}")
print("Reloaded the 100-food model successfully.")

# Helper function to preprocess and predict
# def predict_image(image_path, model, class_names):
#     img = tf.keras.utils.load_img(image_path, target_size=(380, 380))
#     img_array = tf.keras.utils.img_to_array(img)
#     img_array = tf.expand_dims(img_array, 0) # Create a batch

#     preds = model.predict(img_array)
#     pred_class = class_names[np.argmax(preds)]
#     return pred_class

# # Define class names (based on your training data)
# class_names = ['apple', 'banana', 'beef', 'blueberries', 'carrots', 'chicken_wings', 'egg', 'honey', 'mushrooms', 'strawberries']

# # Predict images
# images_to_test = ["/content/burger.jpg", "/content/chicken-wings.jpg", "/content/stake.jpg"]
# for img_p in images_to_test:
#     try:
#         prediction = predict_image(img_p, loaded_model, class_names)
#         print(f"Image: {img_p} | Prediction: {prediction}")
#     except Exception as e:
#         print(f"Error processing {img_p}: {e}")

In [ ]:
# Old 10-food test removed. The notebook now keeps the 100-food model loaded.
print("Using the 100-food model with", len(class_names), "classes")

In [ ]:
# Prediction index is shown by the sample-image cell above.

In [ ]:
# Sample image is shown by the sample-image cell above.

In [ ]:
print("In-memory model:", model_0.evaluate(test_data))
print("Reloaded 100-food model:", loaded_model.evaluate(test_data))

In [ ]:
image = tf.keras.utils.load_img(
    "strawberries.jpg",
    target_size=(380,380)
)

input_arr = tf.keras.utils.img_to_array(image)

input_arr = tf.expand_dims(input_arr, axis=0)

# Using loaded_model which was loaded in previous cells
predictions = loaded_model.predict(input_arr)

predicted_index = np.argmax(predictions)

# Fix: Use the global class_names list instead of train_data.class_names
predicted_food = class_names[predicted_index]

confidence = predictions[0][predicted_index]

print("Food:", predicted_food)
print("Confidence:", confidence)

In [ ]:
pred1 = model_0.predict(input_arr)
pred2 = loaded_model.predict(input_arr)

print(np.argmax(pred1))
print(np.argmax(pred2))

print(np.allclose(pred1, pred2))

In [ ]:
import tensorflow as tf
import numpy as np
import os
from pathlib import Path

# 1. Setup paths and parameters
image_dir = "/content/food_images_100"
model_path = "/content/drive/MyDrive/Nutrify/models/food-100-version.keras"
IMG_SIZE = (380, 380)

# 2. Define class names (Required for 100-class model)
class_names = ['almonds', 'apple', 'apricot_fruit', 'asparagus', 'avocado', 'bacon', 'bagel', 'banana', 'beef_meat', 'beetroot', 'bell_pepper', 'black_beans', 'blackberries', 'blueberries', 'bread', 'broccoli', 'brown_rice', 'butter', 'cabbage', 'cantaloupe_melon', 'carrots', 'cashews', 'cauliflower', 'celery', 'cheddar_cheese', 'cherries', 'chia_seeds', 'chicken_breast', 'chicken_thigh', 'chicken_wings', 'chickpeas', 'coconut', 'cod_fish', 'corn', 'crab', 'cucumber', 'dates_fruit', 'duck_breast', 'egg', 'eggplant', 'figs_fruit', 'garlic', 'grapefruit', 'grapes', 'green_beans', 'ham', 'honey', 'kidney_beans', 'kiwi_fruit', 'lamb_chop', 'lentils', 'lettuce', 'lobster', 'mango_fruit', 'milk', 'mozzarella_cheese', 'mushrooms', 'noodles', 'oats', 'onion', 'orange_fruit', 'papaya_fruit', 'pasta', 'peach_fruit', 'peanut_butter', 'peanuts', 'pear_fruit', 'peas', 'pineapple', 'pistachios', 'plum_fruit', 'pomegranate', 'popcorn', 'pork_chop', 'potato', 'pumpkin', 'pumpkin_seeds', 'radish', 'raspberries', 'rice', 'salmon_fish', 'sardines', 'sesame_seeds', 'shrimp', 'soybeans', 'spinach', 'squid', 'strawberries', 'sunflower_seeds', 'sweet_potato', 'tilapia_fish', 'tofu', 'tomato', 'tuna_fish', 'turkey_breast', 'turnip', 'walnuts', 'watermelon', 'yogurt', 'zucchini']

# 3. Load the model
if os.path.exists(model_path):
    print(f"Loading model from: {model_path}")
    model_100 = tf.keras.models.load_model(model_path)
else:
    print(f"Error: Model file {model_path} not found.")

# 4. Predict on images from the zip directory
if os.path.exists(image_dir):
    print(f"\nPredicting images in {image_dir}:\n" + "-"*50)
    image_paths = list(Path(image_dir).glob("*.jpg"))[:15] # Test first 15 images

    for img_path in image_paths:
        img = tf.keras.utils.load_img(str(img_path), target_size=IMG_SIZE)
        img_array = tf.keras.utils.img_to_array(img)
        img_array = tf.expand_dims(img_array, 0)

        preds = model_100.predict(img_array, verbose=0)
        pred_idx = np.argmax(preds)
        pred_label = class_names[pred_idx]
        confidence = preds[0][pred_idx]

        print(f"File: {img_path.name:25} | Pred: {pred_label:15} | Conf: {confidence:.2%}")
else:
    print(f"Error: Directory {image_dir} not found.")